In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Qwen/Qwen2.5-VL-3B-Instruct",
    local_dir="./qwen_2_5_4b_vl_instruct_weights",
    token="your token here"
)

In [ ]:
!pip install datasets
!pip install qwen-vl-utils[decord]==0.0.8


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
local_dir = snapshot_download(
    repo_id="teknium/OpenHermes-2.5",
    repo_type="dataset",
    local_dir="./sae_dataset"
)

In [ ]:
!nvidia-smi

In [ ]:
# !kill -9 19552

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "./qwen_2_5_4b_vl_instruct_weights", torch_dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("./qwen_2_5_4b_vl_instruct_weights")


In [ ]:
import pandas as pd
df = pd.read_json('/content/sae_dataset/openhermes2_5.json')


In [ ]:
import json
output_file = "dateset_for_sae_3.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    pass

def stream_state_to_disk(prompt_idx,prompt, output_token_id, hidden_state_tensor,output_file):
    # Convert only the CURRENT tensor to a CPU list
    hidden_state_list = hidden_state_tensor.detach().cpu().tolist()

    entry = {
        "prompt_idx": prompt_idx,
        "prompt": prompt,
        "output_token_id":output_token_id,
        "hidden_states": hidden_state_list
    }

    # Append ('a' mode) directly to disk. Memory is instantly freed after closing.
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")

In [ ]:

prompts_for_sae = 1000000

for prompt_idx in range(3000, prompts_for_sae):
  prompt = df.iloc[prompt_idx]['conversations'][0]["value"]

  messages = [
      {
          "role": "user",
          "content": [
              {"type": "text", "text": str(prompt)},
          ],
      }
  ]

  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )


  inputs = processor(
      text=[text],
      padding=True,
      return_tensors="pt",
  )
  inputs = inputs.to("cuda")
  # inputs = inputs.to("cpu")

  output_generation = model.generate(**inputs, max_new_tokens=20000, output_hidden_states=True, return_dict_in_generate=True)

  layer_idx = -1 # using last layer
  batch_selection = -1 # used for taking the last latent repr from multiple kv values for the input prompt as first generation doesn't use kv cache

  for i in range(len(output_generation.hidden_states)):
    output_token_id = int(output_generation.sequences[0][i])

    hidden_state_tensor = output_generation.hidden_states[i][layer_idx][:,batch_selection,:]


    stream_state_to_disk(prompt_idx,prompt,output_token_id,hidden_state_tensor,output_file)

  print(f" done with prompt {prompt_idx}")

  generated_ids = output_generation.sequences
  generated_ids_trimminputsed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  print(output_text)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/dateset_for_sae_3.jsonl /content/drive/MyDrive/

In [ ]:
!cp /content/drive/MyDrive/dateset_for_sae_2.jsonl /content/dateset_for_sae_2.jsonl
!cp /content/drive/MyDrive/dateset_for_sae.jsonl /content/dateset_for_sae.jsonl

In [ ]:
import torch
import torch.nn as nn

class AnthropicStyleSAE(nn.Module):
    def __init__(self, d_model: int, dict_size: int, l1_coefficient: float):
        super().__init__()
        self.d_model = d_model
        self.dict_size = dict_size
        self.l1_coefficient = l1_coefficient

        # Anthropic subtracts a pre-encoder bias, applies weight, then adds encoder bias
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.W_enc = nn.Parameter(torch.nn.init.kaiming_uniform_(torch.empty(d_model, dict_size)))
        self.b_enc = nn.Parameter(torch.zeros(dict_size))

        # Decoder weights (often initialized as transpose of W_enc or normalized)
        self.W_dec = nn.Parameter(torch.nn.init.kaiming_uniform_(torch.empty(dict_size, d_model)))
        # Anthropic typically normalizes decoder weights to unit norm
        self.make_decoder_weights_unit_norm()

    @torch.no_grad()
    def make_decoder_weights_unit_norm(self):
        norms = torch.norm(self.W_dec, dim=1, keepdim=True)
        self.W_dec.data /= (norms + 1e-8)

    def forward(self, x):
        # x shape: [batch_size, d_model]

        # 1. Centering the input around the decoder bias
        x_centered = x - self.b_dec

        # 2. Encode to latent feature activations
        # Anthropic uses a linear layer followed by a ReLU activation
        latent_activations = torch.relu(x_centered @ self.W_enc + self.b_enc)

        # 3. Decode to reconstruct activations
        x_reconstructed = latent_activations @ self.W_dec + self.b_dec

        # 4. Compute Losses
        # Reconstruction loss (MSE)
        reconstruction_loss = torch.mean((x - x_reconstructed) ** 2)

        # L1 Sparsity loss (sum over features, mean over batch)
        # Note: Anthropic often scales this or uses a dynamic L1 coefficient per feature,
        # but a global scalar coefficient is the baseline architecture setup.
        l1_loss = self.l1_coefficient * torch.mean(torch.sum(latent_activations, dim=-1))

        total_loss = reconstruction_loss + l1_loss

        return {
            "x_reconstructed": x_reconstructed,
            "latent_activations": latent_activations,
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "l1_loss": l1_loss
        }

# Quick validation check
# sae = AnthropicStyleSAE(d_model=2048, dict_size=2048*32, l1_coefficient=0.01)


In [ ]:
import pandas as pd

# dataset_for_sae = pd.read_json('/content/dateset_for_sae.jsonl', lines=True)
dataset_for_sae_2 = pd.read_json("/content/dateset_for_sae_2.jsonl", lines=True)

# train_df = pd.concat([dataset_for_sae, dateset_for_sae_2])
train_df = dataset_for_sae_2

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader


column_data = train_df['hidden_states'].to_list()

del train_df

# 2. Convert to a PyTorch float tensor
# This creates a tensor of shape: [total_samples, d_model]
features_tensor = torch.tensor(column_data, dtype=torch.float32)

# 3. Create a TensorDataset wrapper
dataset = TensorDataset(features_tensor)

# 4. Initialize the DataLoader
batch_size = 32  # Define your preferred batch size
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)



In [ ]:
i = 0
for batch in dataloader:
  i += 1
  print(i)
  print(len(batch[0]))

In [ ]:
optimizer = torch.optim.Adam(sae.parameters(), lr=3e-4)

for batch in dataloader:  # batch is a list: [features_tensor_batch]
    optimizer.zero_grad()

    # Unpack the tensor from the batch list/tuple
    x = batch[0]

    outputs = sae(x)
    loss = outputs["loss"]
    print("loss: ", loss.item())

    loss.backward()
    optimizer.step()

    # CRITICAL STEP: Keep the decoder vectors normalized to unit length
    sae.make_decoder_weights_unit_norm()
    print("backprop done one batch")

In [ ]:
from safetensors.torch import save_file
# 1. Extract the dictionary of weights
state_dict = sae.state_dict()

# 2. Save to a file
save_file(state_dict, "qwen_2_5_vl_3b_instruct_sae.safetensors")



In [ ]:
!hf auth login

In [ ]:
!hf upload uninterested-ultramarine/qwen_2_5_vl_3b_instruct_sae_model /content/qwen_2_5_vl_3b_instruct_sae.safetensors


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
local_dir = snapshot_download(
    repo_id="uninterested-ultramarine/qwen_2_5_vl_3b_instruct_sae_model",
    repo_type="model",
    local_dir="./sae_model",
    token="your token here"
)

In [ ]:
from safetensors.torch import load_file
import torch

trained_sae = AnthropicStyleSAE(d_model=2048, dict_size=2048*32, l1_coefficient=0.01).to("cuda")

# Use load_file from safetensors to load the weights
state_dict = load_file("/content/sae_model/qwen_2_5_vl_3b_instruct_sae.safetensors")
trained_sae.load_state_dict(state_dict)

In [ ]:
inter_secting_dims = set()
print(len(inter_secting_dims))

In [ ]:
prompt = "who invented inter preted languages"
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": str(prompt)},
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)


inputs = processor(
    text=[text],
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")
# inputs = inputs.to("cpu")

output_generation = model.generate(**inputs, max_new_tokens=20000, output_hidden_states=True, return_dict_in_generate=True)

layer_idx = -1 # using last layer
batch_selection = -1 # used for taking the last latent repr from multiple kv values for the input prompt as first generation doesn't use kv cache

grouped_top_indices = set()

default_indices = set(output_generation.sequences[0].flatten().tolist())

print("default_indices", default_indices)

for i in range(len(output_generation.hidden_states)):
  output_token_id = int(output_generation.sequences[0][i])
  print("output_token_id ", output_token_id)
  generated_ids = [output_token_id]

  output_text = processor.batch_decode(
      generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  print("decoded_token", output_text)

  hidden_state_tensor = output_generation.hidden_states[i][layer_idx][:,batch_selection,:]

  with torch.no_grad():
    outputs = trained_sae(hidden_state_tensor)
    top_k_for_this = torch.topk(outputs["latent_activations"],k=32,dim=-1).indices
    grouped_top_indices.update(top_k_for_this.flatten().tolist())
    # print("sae latent repr shape", outputs["latent_activations"].shape)
    # print("argmax outputs sae", )
    # print("sae outputs", outputs["latent_activations"])

# print(f" done with prompt {prompt_idx}")

print(grouped_top_indices)
inter_secting_dims = grouped_top_indices if len(inter_secting_dims)==0 else grouped_top_indices & inter_secting_dims

In [ ]:
print(default_indices)

In [ ]:
print(default_indices)

In [ ]:
inter_secting_dims = inter_secting_dims-default_indices
print(inter_secting_dims)

In [37]:

prompts = [
    # 1. Origins and history of programming languages
    "Who invented interpreted programming languages?",
    "Who pioneered compiled programming languages?",
    "When did high-level programming languages first emerge?",
    "What motivated the creation of Python?",
    "Why was the C programming language developed?",
    "Who designed JavaScript and what problem did it solve?",
    "How did programming languages evolve from machine code?",
    "What was the first widely used interpreted language?",
    "Why were early programming languages difficult to use?",
    "How did the history of compilers shape modern software?",

    # 2. Interpreted versus compiled execution
    "Why is Python generally considered an interpreted language?",
    "Why is C traditionally described as a compiled language?",
    "What distinguishes an interpreter from a compiler?",
    "Can an interpreted language be compiled before execution?",
    "Does Python compile source code into bytecode?",
    "Why does C usually require compilation before running?",
    "How does an interpreter execute a program?",
    "What happens between writing code and executing it?",
    "Can the same language support interpreted and compiled execution?",
    "Why is the distinction between compiled and interpreted languages imperfect?",

    # 3. Runtime execution and virtual machines
    "What happens when the Python interpreter runs a script?",
    "How does the Java Virtual Machine execute bytecode?",
    "Why does Java use an intermediate bytecode representation?",
    "What is the role of a language runtime?",
    "How does CPython execute Python bytecode?",
    "What happens when a program encounters an instruction at runtime?",
    "Why do some languages require a virtual machine?",
    "How does just-in-time compilation work?",
    "What is the difference between bytecode and machine code?",
    "How does a runtime translate program instructions into CPU operations?",

    # 4. Performance and efficiency
    "Why is C often faster than Python?",
    "Why can interpreted programs have higher execution overhead?",
    "How does compilation improve program performance?",
    "Why does Python sometimes run slowly on computational workloads?",
    "Can just-in-time compilation make Python faster?",
    "What makes native machine code efficient?",
    "Why do compiled programs sometimes start faster than interpreted programs?",
    "How does runtime optimization affect execution speed?",
    "When can an interpreted language outperform a compiled one?",
    "What determines the performance of a programming language?",

    # 5. Language design and tradeoffs
    "Why was Python designed to emphasize readability?",
    "Why does C expose low-level memory operations?",
    "How do programming language designers balance simplicity and control?",
    "Why does JavaScript behave differently from Python?",
    "What tradeoffs exist between developer productivity and execution speed?",
    "Why do some programming languages enforce static typing?",
    "How does language design influence software maintainability?",
    "Why do scripting languages often favor concise syntax?",
    "What makes a programming language suitable for systems programming?",
    "How do language designers decide which features to include?",

    # 6. Syntax, semantics, and language rules
    "What is the difference between programming language syntax and semantics?",
    "Why does Python use indentation to define code blocks?",
    "Why does C use braces to delimit blocks?",
    "How does JavaScript handle automatic type conversion?",
    "What makes a programming language dynamically typed?",
    "How does static typing affect error detection?",
    "Why can two languages express the same algorithm differently?",
    "What is the difference between a syntax error and a runtime error?",
    "How do language specifications define program behavior?",
    "Why can valid code produce unexpected results?",

    # 7. Compilers and compilation stages
    "What are the main stages of a compiler?",
    "How does a compiler transform source code into machine code?",
    "What does a lexical analyzer do?",
    "Why do compilers construct abstract syntax trees?",
    "How does semantic analysis detect programming errors?",
    "What is the purpose of an intermediate representation in a compiler?",
    "How does compiler optimization reduce execution cost?",
    "What is the difference between ahead-of-time and just-in-time compilation?",
    "How does a linker combine compiled object files?",
    "Why can compiler optimizations change the performance of identical algorithms?",

    # 8. Programming language ecosystems
    "Why is Python popular in machine learning?",
    "Why is C widely used in operating systems?",
    "Why is JavaScript dominant in browser programming?",
    "How does Python call functions implemented in C?",
    "Why do numerical Python libraries use native extensions?",
    "How do programming languages interact through foreign function interfaces?",
    "Why do developers write performance-critical code in multiple languages?",
    "What role does C play beneath higher-level programming languages?",
    "How does a Python package execute native machine code?",
    "Why do programming ecosystems depend on compilers and runtimes?",

    # 9. Portability, memory, and hardware
    "Why can the same Python program run on different operating systems?",
    "Why might a compiled binary fail on another processor architecture?",
    "How does compilation depend on the target hardware?",
    "What is the difference between source portability and binary portability?",
    "Why does C give programmers direct access to memory?",
    "How does garbage collection affect programming language design?",
    "Why do memory management rules differ between C and Python?",
    "How does an operating system load an executable program?",
    "What role does an application binary interface play in execution?",
    "Why can hardware architecture influence compiler output?",

    # 10. Informal wording, misconceptions, and edge cases
    "Why is Python called interpreted when it produces bytecode?",
    "Is C always faster than Python in real applications?",
    "Can a compiler exist for a language usually called interpreted?",
    "Why do people casually describe Java as both compiled and interpreted?",
    "Does an interpreter translate an entire program before running it?",
    "Why is calling JavaScript a scrappy scripting language misleading?",
    "Are interpreted languages necessarily easier to debug?",
    "Can compiled languages support interactive execution?",
    "Why is the compiled-versus-interpreted distinction sometimes confusing?",
    "If Python and C can perform the same computation, why execute them differently?"
]


In [42]:

intersecting_dims = None

for prompt in prompts:
  messages = [
      {
          "role": "user",
          "content": [
              {"type": "text", "text": str(prompt)},
          ],
      }
  ]

  text = processor.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )


  inputs = processor(
      text=[text],
      padding=True,
      return_tensors="pt",
  )
  inputs = inputs.to("cuda")
  # inputs = inputs.to("cpu")

  output_generation = model.generate(**inputs, max_new_tokens=20000, output_hidden_states=True, return_dict_in_generate=True)

  layer_idx = -1 # using last layer
  batch_selection = -1 # used for taking the last latent repr from multiple kv values for the input prompt as first generation doesn't use kv cache

  grouped_top_indices = set()

  generated_ids = []

  for i in range(len(output_generation.hidden_states)):
    output_token_id = int(output_generation.sequences[0][i])
    # print("output_token_id ", output_token_id)
    generated_ids.append(output_token_id)


    # print("decoded_token", output_text)

    hidden_state_tensor = output_generation.hidden_states[i][layer_idx][:,batch_selection,:]

    with torch.no_grad():
      outputs = trained_sae(hidden_state_tensor)
      top_k_for_this = torch.topk(outputs["latent_activations"],k=32,dim=-1).indices
      grouped_top_indices.update(top_k_for_this.flatten().tolist())
      # print("sae latent repr shape", outputs["latent_activations"].shape)
      # print("argmax outputs sae", )
      # print("sae outputs", outputs["latent_activations"])

# print(f" done with prompt {prompt_idx}")

  output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
  print("output_text",output_text)

  print(grouped_top_indices)
  if intersecting_dims is None:
        intersecting_dims = grouped_top_indices
  else:
      intersecting_dims &= grouped_top_indices

  print("inter_secting_dims",intersecting_dims)
  print("inter_secting_dims len",len(intersecting_dims))

output_text ['system\nYou are a helpful assistant.\nuser\nWho invented interpreted programming languages?\nassistant\nInterpreted programming languages were invented in the 1960s and 1970s by a number of computer scientists, including:\n\n  * John McCarthy, who developed LISP in the 1950s.\n  * Alan Kay, who developed Smalltalk in the 1970s.\n  * Dennis Ritchie, who developed C in the 1970s.\n\nThese languages were designed to be more flexible and easier to use than compiled languages, which require the programmer to write code that is then translated into machine language before it can be executed. Interpreted languages allow the programmer']
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 16391, 11, 13, 12, 14, 15, 16, 17, 19, 18, 10254, 57381, 63526, 10283, 14382, 61501, 12350, 36931, 32839, 59464, 51286, 6230, 36962, 12412, 32894, 53377, 63626, 61579, 43160, 22689, 53411, 37032, 63680, 26821, 49358, 39118, 47312, 4305, 16599, 41176, 39132, 45279, 8415, 49379, 12533, 35064, 63742, 63748, 45319, 